# 04 · The heavy steps on Euler: QC, kallisto and STAR
**Goal:** run the compute-heavy steps (full-read QC, trimming, kallisto, STAR) on ETH's cluster instead of the laptop, and copy back the small results.
**Input:** the 27 `.sra` runs and the GENCODE vM25 reference (downloaded on Euler). **Output** (copied back): `results/kallisto/`, `results/qc/` and `results/star/`. STAR counts (`results/star/counts_unstranded.csv.gz`) are in git, so this notebook can be skipped.
**Kernel:** Python 3 (laptop part) · **Time:** the STAR array took about 2 h on Euler (measured); the FastQC + kallisto array has not been timed yet.

Notebook 02 does the QC and kallisto steps on the laptop (hours, fan at full speed); this notebook does them, and STAR, on **Euler**, ETH's computing cluster. Jobs are scheduled with **SLURM**: you submit them from a *login node* and they run on *compute nodes*. STAR aligns each read to its exact position in the genome and counts reads per gene; its genome index needs ~30 GB of RAM, hence the cluster. Every run goes through the same fastp trimming as on the laptop, so kallisto and STAR see identical reads.

| Step | Script | Resources |
|---|---|---|
| Software environment (versions pinned) | `euler/00_setup.sh` | login node |
| Download genome, annotation, transcripts, 27 runs | `euler/01_download.sh` (in `tmux`) | ~55 GB; time depends on NCBI |
| kallisto index | `euler/02b_kallisto_index.sh` | 1 core, 8 GB |
| STAR index | `euler/02_star_index.sh` | 10 cores, 60 GB; took 22 min |
| **Light array:** FastQC (all reads) + fastp + kallisto | `euler/03_process_array.sh`, `STAGES=qc+kallisto` | 27 tasks, 8 cores × 2 GB |
| **Heavy array:** fastp + STAR | `euler/03_process_array.sh`, `STAGES=star` | 27 tasks, 8 cores × 6 GB; 3.5–13 min per run |
| Pack results | `euler/04_collect.sh` | three tarballs |

`submit.sh` submits both indices and both arrays with the right dependencies. Stages whose output exists are skipped, so a rerun does only what is missing. *The kallisto and FastQC additions were tested locally (control flow, with a stand-in for STAR) but have not been run on Euler yet.*

## On Euler (ETH network or VPN)
```bash
ssh <user>@euler.ethz.ch
git clone https://github.com/Jeffateth/RNAseq_aging         # first time; later: cd RNAseq_aging && git pull
bash RNAseq_aging/euler/00_setup.sh                          # once; adds kallisto + FastQC to the environment, pins versions
tmux new -s dl                                               # tmux keeps the download alive if you disconnect
bash RNAseq_aging/euler/01_download.sh                       # resumable; leave with Ctrl-b then d
bash RNAseq_aging/euler/submit.sh                            # indices, then the light and heavy arrays
squeue -u $USER                                              # PD = pending, R = running
bash RNAseq_aging/euler/04_collect.sh                        # when all tasks are COMPLETED
```
If `git pull` complains about local changes to `euler/` (e.g. edits made by hand earlier), run `git checkout euler/` first. Scratch storage deletes files after 15 days: copy results back promptly. From a terminal on **your laptop** (not Euler):
```bash
mkdir -p results/euler && scp '<user>@euler.ethz.ch:/cluster/scratch/<user>/rnaseq_aging/*_results.tar.gz' results/euler/
```
Pitfalls met on the way: batch jobs do not inherit `conda` from your login shell (fixed in `euler/common.sh`); a per-user memory limit lets only about 2 STAR tasks run at once, which is why the light array asks for little memory.

## On the laptop: unpack and load the results

In [1]:
import sys; sys.path.insert(0, '../scripts')
import helpers as h
runs = h.load_runs()
# kallisto_results -> results/kallisto, qc_results -> results/qc, star_results -> results/star/mapping (missing tarballs are skipped)
h.unpack_euler()

not found: kallisto_results.tar.gz
not found: qc_results.tar.gz
unpacked star_results.tar.gz -> results/star


### kallisto and QC
The same checks as notebook 02: every run quantified from trimmed reads, pseudoaligned share 60–80 %.

In [2]:
# downloaded / quantified / trimmed per run; raises if any run was processed with a different trimming setting
h.check_consistent(runs, True)
h.status(runs)

consistent: all quantified runs used trimmed=True


,group,sra_GB,quantified,trimmed
SRR,,,,
SRR21355401,aging 3mo,1.67,True,True
SRR21355396,aging 3mo,3.03,True,True
SRR21355392,aging 3mo,1.88,True,True
SRR21355388,aging 3mo,1.94,True,True
SRR21355398,aging 18mo,0.78,True,True
SRR21355394,aging 18mo,0.95,True,True
SRR21355390,aging 18mo,3.23,True,True
SRR21355386,aging 18mo,1.13,True,True
SRR21355397,aging 21mo,1.66,True,True


In [3]:
# per-run read pairs and pseudoaligned share; combined FastQC report (open the HTML) and verdict table
print('report:', h.multiqc())
h.qc_summary(runs)

report: <repo>/results/qc/multiqc_raw.html


,pairs_M,aligned_%,q30_%,dup_%,group
SRR21355401,14.9,63.0,89.7,7.6,aging 3mo
SRR21355396,25.7,64.6,87.4,6.9,aging 3mo
SRR21355392,16.1,64.8,88.2,6.6,aging 3mo
SRR21355388,16.9,65.0,88.5,7.2,aging 3mo
SRR21355398,6.4,65.2,86.5,3.3,aging 18mo
SRR21355394,7.8,65.3,86.2,2.9,aging 18mo
SRR21355390,28.2,61.9,88.7,8.0,aging 18mo
SRR21355386,9.3,65.5,86.1,3.3,aging 18mo
SRR21355397,14.2,64.1,88.5,5.6,aging 21mo
SRR21355393,9.8,67.6,87.7,3.8,aging 21mo


### STAR
`counts` = genes × runs (unstranded column of STAR's ReadsPerGene); `qc` = mapping statistics. **Checks:** `unique_%` = read pairs mapped to a single genomic location (expect ≈ 80–90 %); `multi_%` = mapped to several places; `assigned_to_gene_%` = counted in a gene; `frac_read1_same_strand` ≈ 0.5 confirms the libraries are **unstranded**, so the unstranded count column is the right one.

In [4]:
# parse STAR's per-run outputs (from the unpacked tarball)
counts, qc = h.load_star(runs)
print(counts.shape, 'genes x runs')
qc

(55401, 27) genes x runs


,input_pairs_M,unique_%,multi_%,assigned_to_gene_%,frac_read1_same_strand,group
SRR,,,,,,
SRR21355401,14.89,80.42,6.89,52.33,0.5,aging 3mo
SRR21355396,25.70,80.53,7.24,53.74,0.5,aging 3mo
SRR21355392,16.08,80.62,7.12,53.73,0.5,aging 3mo
SRR21355388,16.86,80.12,7.11,53.93,0.5,aging 3mo
SRR21355398,6.39,79.35,7.72,53.62,0.5,aging 18mo
SRR21355394,7.83,82.33,6.41,54.69,0.5,aging 18mo
SRR21355390,28.21,79.58,7.57,51.16,0.5,aging 18mo
SRR21355386,9.34,81.77,6.57,54.75,0.5,aging 18mo
SRR21355397,14.17,81.62,6.82,53.71,0.5,aging 21mo


In [5]:
# stop if the library looks stranded; save the count matrix and the QC table
assert counts.shape[1] == 27
assert qc.frac_read1_same_strand.between(0.4, 0.6).all(), 'library looks stranded: use another ReadsPerGene column'
counts.to_csv(h.STAR / 'counts_unstranded.csv.gz')
qc.to_csv(h.STAR / 'star_qc.csv')
print('saved results/star/counts_unstranded.csv.gz')

saved results/star/counts_unstranded.csv.gz


## Mapping rates: STAR vs kallisto
The two numbers are not the same quantity (unique genomic mapping vs pseudoalignment to transcripts) but should be of similar size.

In [6]:
# STAR unique / multi / assigned rates next to kallisto's pseudoaligned rate
k = h.qc_summary(runs)[['aligned_%']].rename(columns={'aligned_%': 'kallisto_pseudoaligned_%'})
qc[['unique_%', 'multi_%', 'assigned_to_gene_%']].join(k).join(runs['group'])

,unique_%,multi_%,assigned_to_gene_%,kallisto_pseudoaligned_%,group
SRR,,,,,
SRR21355401,80.42,6.89,52.33,63.0,aging 3mo
SRR21355396,80.53,7.24,53.74,64.6,aging 3mo
SRR21355392,80.62,7.12,53.73,64.8,aging 3mo
SRR21355388,80.12,7.11,53.93,65.0,aging 3mo
SRR21355398,79.35,7.72,53.62,65.2,aging 18mo
SRR21355394,82.33,6.41,54.69,65.3,aging 18mo
SRR21355390,79.58,7.57,51.16,61.9,aging 18mo
SRR21355386,81.77,6.57,54.75,65.5,aging 18mo
SRR21355397,81.62,6.82,53.71,64.1,aging 21mo
